In [ ]:
BASIN = "calpella"
RUN_NAME = "fake_shift4y_dry_peaky"
SHIFT_YEARS = 4
TARGET_COL = "NR CALPELLA FLOW COE CPL"
LOW_Q, HIGH_Q = 0.10, 0.95
LOW_MULT, HIGH_MULT, MID_MULT = 0.80, 1.25, 1.00
APPLY_DRY_MONTHS = True
DRY_MONTHS = [5, 6, 7, 8, 9, 10]
DRY_MONTHS_MULT = 0.90

EXTRA_COLS = [
    "EF RUSSIAN 20 ET-POTENTIAL RUN:BASIN AVERAGE 60 YR",
    "EF RUSSIAN 20 PRECIP-INC SCREENED",
    "POTTER VALLEY CA FLOW USGS_ADJUSTED",
    "UKIAH CA HUMIDITY USAF-NOAA",
    "UKIAH CA SOLAR RADIATION USAF-NOAA",
    "UKIAH CA TEMPERATURE USAF-NOAA",
    "UKIAH CA WINDSPEED USAF-NOAA",
]

In [ ]:
import os, sys, numpy as np, pandas as pd
from pathlib import Path
library_path = os.path.join('..', '..', '..', 'UCB-USACE-RR-PROJECT')
sys.path.insert(0, library_path)
from UCB_training.UCB_utils import data_dir, basin_prefix, read_hec_table, write_hec_table, find_label

In [ ]:
DATA_DIR = data_dir()
MAIN_SRC = DATA_DIR / "daily_shift.csv"
PHYS_SRC = DATA_DIR / f"{basin_prefix(BASIN)}_daily_shift.csv"
if not MAIN_SRC.exists(): raise FileNotFoundError(MAIN_SRC)
if not PHYS_SRC.exists(): raise FileNotFoundError(PHYS_SRC)
main_df, main_meta = read_hec_table(MAIN_SRC)
phys_df, phys_meta = read_hec_table(PHYS_SRC)
main_date = find_label(main_df, "Date")
phys_date = find_label(phys_df, "Date")
phys_day = None
try:
    phys_day = find_label(phys_df, "Day")
except Exception:
    pass

In [ ]:
d = pd.to_datetime(main_df[main_date], format="%d-%b-%y", errors="coerce")
if d.isna().all(): raise RuntimeError("MAIN dates parse failed")
t_col = find_label(main_df, TARGET_COL)
t = pd.to_numeric(main_df[t_col], errors="coerce").to_numpy(float)
finite = np.isfinite(t)
if finite.any():
    ql = np.nanpercentile(t[finite], LOW_Q*100)
    qh = np.nanpercentile(t[finite], HIGH_Q*100)
    y = t.copy(); lows=y<=ql; highs=y>=qh; mids=~(lows|highs)
    y[lows]*=LOW_MULT; y[highs]*=HIGH_MULT; y[mids]*=MID_MULT; y=np.maximum(y,0)
    main_df[t_col] = [f"{v:.6f}" if "." in str(o) else str(int(round(v))) if str(o).strip().isdigit() else str(v)
                      for o,v in zip(main_df[t_col].astype(str), y)]
if APPLY_DRY_MONTHS:
    dry = d.dt.month.isin(DRY_MONTHS).to_numpy()
    for name in EXTRA_COLS:
        try:
            c = find_label(main_df, name)
        except Exception:
            continue
        a = pd.to_numeric(main_df[c], errors="coerce").to_numpy(float)
        b = np.where(np.isfinite(a) & dry, a*DRY_MONTHS_MULT, a)
        main_df[c] = [str(v) if np.isfinite(v) else main_df[c].iloc[i] for i,v in enumerate(b)]

In [ ]:
shifted = (d + pd.DateOffset(years=SHIFT_YEARS)).dt.strftime("%d-%b-%y")
main_df[main_date] = shifted
pd_phys = pd.to_datetime(phys_df[phys_date], format="%d-%b-%y", errors="coerce")
if pd_phys.isna().all(): raise RuntimeError("PHYS dates parse failed")
shifted_phys = (pd_phys + pd.DateOffset(years=SHIFT_YEARS)).dt.strftime("%d-%b-%y")
phys_df[phys_date] = shifted_phys
if phys_day is not None:
    phys_df[phys_day] = shifted_phys
OUT_DIR = DATA_DIR / "synthetic" / BASIN / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAIN_OUT = OUT_DIR / "daily_shift.csv"
PHYS_OUT = OUT_DIR / "physics_daily.csv"
write_hec_table(main_df, main_meta, MAIN_OUT)
write_hec_table(phys_df, phys_meta, PHYS_OUT)
smin = pd.to_datetime(shifted, format="%d-%b-%y", errors="coerce").min().date()
smax = pd.to_datetime(shifted, format="%d-%b-%y", errors="coerce").max().date()
print("Saved:\n ", MAIN_OUT, "\n ", PHYS_OUT)
print("Shifted window:", smin, "→", smax)